In [ ]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models_luke.adasyn.models import ADASYNModel

In [ ]:
# Colab setup — run this cell first each session
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/Katabatic'
    sys.path.insert(0, REPO)
    os.chdir(REPO)
    # Install deps — comment out after first run to save time
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
    print('Colab ready. CWD:', os.getcwd())
except ImportError:
    pass  # Running locally — no setup needed

In [ ]:
ADASYN = lambda: ADASYNModel(
    sampling_strategy="auto",
    n_neighbors=5,
    random_state=42,
)

In [ ]:
# Preprocess all datasets
DATASETS = ["car", "adult", "magic", "shuttle", "nursery"]

for dataset in DATASETS:
    dataset_path = ROOT / "raw_data" / f"{dataset}.csv"
    output_path = ROOT / "discretized_data" / f"{dataset}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Preprocessing {dataset}...")
    discretize_preprocess(str(dataset_path), str(output_path))

In [ ]:
# Run pipeline on all datasets
for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"ADASYN -> {dataset}")
    input_csv = str(ROOT / "discretized_data" / f"{dataset}.csv")
    output_dir = str(ROOT / "sample_data" / dataset)
    real_test_dir = output_dir
    synthetic_dir = str(ROOT / "synthetic" / dataset / "adasyn")

    pipeline = TrainTestSplitPipeline(model=ADASYN)
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=real_test_dir,
    )
    print(result)